# 03.5_model.ipynb — SVM Importance and Within-Subject CV

**Purpose:** Two targeted fixes for the cross-subject RF generalization failure found in 03_model.

**Option 3 — SVM permutation importance (LOSO):** RF does not exceed the null cross-subject. SVM does (0.558 valence, 0.538 arousal). Compute permutation importance using SVM as the model instead, giving interpretability vectors backed by the classifier that actually generalizes. Used for RQ2 (FAA) and RQ5 (dominance) claims.

**Option 1 — Within-subject 5-fold CV:** Trains and tests on the same subject, eliminating distribution shift. Validates that EEG features contain real signal and importance vectors are meaningful rather than noise. Provides within-subject accuracy as the upper-bound reference for RQ3 (LOSO generalization gap).

**Inputs:** `results/X_features.npy`, `results/y.npy`, `results/subject_ids.npy`, `results/loso_results.pkl`

**Outputs:** `results/svm_importance.pkl`, `results/within_subject.pkl`

**Estimated runtime:** 45-90 min total. No null distribution run needed here.

In [1]:
import sys
sys.path.insert(0, '../src')

import numpy as np
import pickle
from pathlib import Path

from models import (
    run_loso_svm_importance, run_within_subject,
    aggregate_importance, importance_correlation,
    TARGETS, BAND_NAMES, CORR_KEYS,
)

# Load features — float32 for speed
X_features  = np.load('../results/X_features.npy').astype(np.float32)
y           = np.load('../results/y.npy')           # keep float64 for median
subject_ids = np.load('../results/subject_ids.npy')

# Load LOSO results from 03_model for comparison
with open('../results/loso_results.pkl', 'rb') as f:
    loso_results = pickle.load(f)
with open('../results/importance_corr.pkl', 'rb') as f:
    rf_loso_corr = pickle.load(f)

print(f"X_features: {X_features.shape}  dtype={X_features.dtype}")
print(f"Loaded LOSO results and RF importance correlation from 03_model.")

X_features: (1280, 131)  dtype=float32
Loaded LOSO results and RF importance correlation from 03_model.


## SVM Permutation Importance (LOSO)

32 folds, EEG only, SVM fit + permutation importance on test fold.
Estimated runtime: 30-50 min.

In [2]:
svm_imp = run_loso_svm_importance(
    X_eeg        = X_features,
    y            = y,
    subject_ids  = subject_ids,
    n_perm_repeats = 20,
    n_jobs       = -3,
)
print("SVM importance loop complete.")

SVM importance folds:   0%|          | 0/32 [00:00<?, ?fold/s]

SVM importance loop complete.


In [3]:
# Band-level importance — SVM (primary RQ1 result, now backed by cross-subject classifier)
print("SVM band-level importance (summed across 32 electrodes, averaged across 32 folds):")
print(f"{'Band':<8}", '  '.join(f"{t:>10}" for t in TARGETS))
print('-' * 44)
for b, band in enumerate(BAND_NAMES):
    row = '  '.join(
        f"{svm_imp['importance_agg'][t]['band'][b]:>10.5f}"
        for t in TARGETS
    )
    print(f"{band:<8}  {row}")

SVM band-level importance (summed across 32 electrodes, averaged across 32 folds):
Band        valence     arousal   dominance
--------------------------------------------
theta       -0.00816    -0.00469     0.00680
alpha       -0.01652    -0.00961    -0.00527
beta        -0.01305    -0.00801     0.00645
gamma       -0.01195    -0.01820     0.01020


In [4]:
# RQ5: SVM importance correlation vs RF importance correlation
print("Pairwise Spearman importance correlation — SVM vs RF (LOSO):")
print(f"{'Pair':<30} {'SVM rho':>8} {'RF rho':>8}  {'Consistent?':>12}")
print('-' * 65)
for key in CORR_KEYS:
    svm_rho = svm_imp['importance_corr'][key]['rho']
    rf_rho  = rf_loso_corr[key]['rho']
    # Consistent if both near zero (dominance pairs) or both positive (valence-arousal)
    consistent = 'YES' if abs(svm_rho - rf_rho) < 0.15 else 'CHECK'
    print(f"{key:<30} {svm_rho:>8.3f} {rf_rho:>8.3f}  {consistent:>12}")

print("\nIf SVM and RF dominance correlations both near zero → robust RQ5 evidence.")
print("If they diverge → importance structure is model-dependent, treat as exploratory.")

Pairwise Spearman importance correlation — SVM vs RF (LOSO):
Pair                            SVM rho   RF rho   Consistent?
-----------------------------------------------------------------
valence_arousal                   0.097    0.124           YES
valence_dominance                 0.031   -0.010           YES
arousal_dominance                -0.260    0.006         CHECK

If SVM and RF dominance correlations both near zero → robust RQ5 evidence.
If they diverge → importance structure is model-dependent, treat as exploratory.


In [5]:
# Save SVM importance results
with open('../results/svm_importance.pkl', 'wb') as f:
    pickle.dump(svm_imp, f)

for target in TARGETS:
    np.save(f'../results/svm_importance_mean_{target}.npy',
            svm_imp['importance_mean'][target])

print("Saved: svm_importance.pkl, svm_importance_mean_{valence,arousal,dominance}.npy")

Saved: svm_importance.pkl, svm_importance_mean_{valence,arousal,dominance}.npy


## Within-Subject 5-Fold CV

160 total folds (32 subjects × 5), parallelized across subjects.
Trains and tests within the same subject — no cross-subject distribution shift.
Estimated runtime: 15-30 min.

In [7]:
ws = run_within_subject(
    X_eeg           = X_features,
    y               = y,
    subject_ids     = subject_ids,
    n_splits        = 5,
    n_perm_repeats  = 20,
    rf_n_estimators = 150,
    n_jobs          = -3,
)
print("Within-subject CV complete.")

Within-subject CV:   0%|          | 0/32 [00:00<?, ?subject/s]

Within-subject CV complete.


/Users/shlok/Desktop/eegec-deap/notebooks/../src/models.py:541: RuntimeWarning: Mean of empty slice
  float(np.nanmean(sr['metrics'][t][model]['acc'])))
/Users/shlok/Desktop/eegec-deap/notebooks/../src/models.py:543: RuntimeWarning: Mean of empty slice
  float(np.nanmean(sr['metrics'][t][model]['f1'])))


In [8]:
# How many subjects were usable per target per model
for target in TARGETS:
    for model in ('svm', 'rf'):
        acc_arr = np.array(ws['metrics'][target][model]['acc'])
        n_valid = int(np.sum(~np.isnan(acc_arr)))
        mean_acc = float(np.nanmean(acc_arr))
        print(f"{target:<12} {model:<5} valid_subjects={n_valid}/32  acc={mean_acc:.3f}")

valence      svm   valid_subjects=32/32  acc=0.545
valence      rf    valid_subjects=32/32  acc=0.548
arousal      svm   valid_subjects=32/32  acc=0.556
arousal      rf    valid_subjects=32/32  acc=0.562
dominance    svm   valid_subjects=31/32  acc=0.569
dominance    rf    valid_subjects=31/32  acc=0.557


In [13]:
# Within-subject accuracy vs LOSO accuracy (RQ3: generalization gap)
print("Within-subject vs LOSO accuracy (EEG, mean across subjects):")
print(f"{'Target':<12} {'Model':<6} {'Within-subj':>12} {'LOSO':>8} {'Gap':>8}")
print('-' * 52)
for target in TARGETS:
    for model in ('svm', 'rf'):
        ws_acc   = np.nanmean(ws['metrics'][target][model]['acc'])
        loso_acc = np.nanmean(loso_results['metrics'][target]['eeg'][model]['acc'])
        gap      = ws_acc - loso_acc
        print(f"{target:<12} {model:<6} {ws_acc:>12.3f} {loso_acc:>8.3f} {gap:>8.3f}")

print("\nLarger gap for valence vs arousal → valence more neurally idiosyncratic (RQ3).")

Within-subject vs LOSO accuracy (EEG, mean across subjects):
Target       Model   Within-subj     LOSO      Gap
----------------------------------------------------
valence      svm           0.545    0.558   -0.013
valence      rf            0.548    0.527    0.021
arousal      svm           0.556    0.500    0.056
arousal      rf            0.562    0.537    0.024
dominance    svm           0.569    0.471    0.098
dominance    rf            0.557    0.484    0.073

Larger gap for valence vs arousal → valence more neurally idiosyncratic (RQ3).


In [10]:
# Within-subject importance correlation (RQ5 validation)
# If dominance orthogonality holds within-subject too, result is robust
print("Within-subject importance correlation (RQ5 robustness check):")
print(f"{'Pair':<30} {'WS-SVM':>8} {'WS-RF':>8} {'LOSO-RF':>8}")
print('-' * 60)
for key in CORR_KEYS:
    ws_svm  = ws['importance_corr']['svm'][key]['rho']
    ws_rf   = ws['importance_corr']['rf'][key]['rho']
    loso_rf = rf_loso_corr[key]['rho']
    print(f"{key:<30} {ws_svm:>8.3f} {ws_rf:>8.3f} {loso_rf:>8.3f}")

print("\nIf dominance pairs near zero across all three columns → strongest possible RQ5 evidence.")

Within-subject importance correlation (RQ5 robustness check):
Pair                             WS-SVM    WS-RF  LOSO-RF
------------------------------------------------------------
valence_arousal                  -0.097   -0.193    0.124
valence_dominance                -0.055    0.017   -0.010
arousal_dominance                -0.275   -0.122    0.006

If dominance pairs near zero across all three columns → strongest possible RQ5 evidence.


In [11]:
# Within-subject band-level importance (RF and SVM)
for model in ('svm', 'rf'):
    print(f"\nWithin-subject band importance ({model.upper()}):")
    print(f"{'Band':<8}", '  '.join(f"{t:>10}" for t in TARGETS))
    print('-' * 44)
    for b, band in enumerate(BAND_NAMES):
        row = '  '.join(
            f"{ws['importance_agg'][t][model]['band'][b]:>10.5f}"
            for t in TARGETS
        )
        print(f"{band:<8}  {row}")


Within-subject band importance (SVM):
Band        valence     arousal   dominance
--------------------------------------------
theta       -0.02789     0.07465    -0.03927
alpha       -0.03809     0.05641    -0.01976
beta        -0.01922     0.00656     0.01399
gamma       -0.02566     0.01605    -0.00859

Within-subject band importance (RF):
Band        valence     arousal   dominance
--------------------------------------------
theta       -0.01625    -0.02605    -0.03859
alpha       -0.01203    -0.02992    -0.04710
beta        -0.00676    -0.01250    -0.02536
gamma       -0.01352    -0.01910    -0.00690


In [12]:
# Save within-subject results
with open('../results/within_subject.pkl', 'wb') as f:
    pickle.dump(ws, f)

print("Saved: within_subject.pkl")
print("\nAll 03.5 outputs saved. Ready for 04_results.ipynb.")

Saved: within_subject.pkl

All 03.5 outputs saved. Ready for 04_results.ipynb.
